<a href="https://colab.research.google.com/github/herr-hollis/F1-Bahrain-2026-Power-BI/blob/main/F1_Team_Radio_(More_Data).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
STEP = "transcribe"         # "download" | "transcribe" | "both"
SESSION = "Race"            # or "Qualifying"
MODEL = "small"             # "base" = faster, "medium" = more accurate

import os
import re
import sys
import time
import shutil
import zipfile
import importlib
import site
import subprocess
from collections import Counter

import pandas as pd
import requests

OPENF1 = "https://api.openf1.org/v1"
CLIPS = "radio_clips"
OUT = "data"
os.makedirs(OUT, exist_ok=True)


def pip_import(pkg, module):
    try:
        return importlib.import_module(module)
    except ModuleNotFoundError:
        print(f"Installing {pkg} (one-time)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        importlib.invalidate_caches()
        if site.getusersitepackages() not in sys.path:
            sys.path.append(site.getusersitepackages())
        return importlib.import_module(module)


def of1(endpoint, **params):
    q = "&".join(f"{k}={v}" for k, v in params.items())
    time.sleep(0.4)
    r = requests.get(f"{OPENF1}/{endpoint}?{q}", timeout=60)
    if r.status_code == 404:
        return pd.DataFrame()
    r.raise_for_status()
    return pd.DataFrame(r.json())


# ======================================================================= STEP 1: download clips
def download():
    ss = of1("sessions", year=2026)
    ss["date_start"] = pd.to_datetime(ss["date_start"], utc=True, format="ISO8601")
    ss = ss[(ss["date_start"] >= "2026-10-01") & (ss["date_start"] <= "2026-10-06")
            & (ss["session_name"] == SESSION)]
    if ss.empty:
        raise SystemExit(f"OpenF1 has no {SESSION} session for Sepang yet.")
    sk = int(ss.iloc[0]["session_key"])

    drivers = of1("drivers", session_key=sk).drop_duplicates("driver_number")
    acr = dict(zip(drivers["driver_number"], drivers["name_acronym"]))

    radio = of1("team_radio", session_key=sk)
    if radio.empty:
        raise SystemExit("No team radio clips published for this session.")
    radio["date"] = pd.to_datetime(radio["date"], utc=True, format="ISO8601")
    print(f"{len(radio)} radio clips across {radio['driver_number'].nunique()} drivers")

    laps = of1("laps", session_key=sk)
    if not laps.empty:
        laps["date_start"] = pd.to_datetime(laps["date_start"], utc=True, format="ISO8601")
        radio = pd.merge_asof(radio.sort_values("date"),
                              laps.dropna(subset=["date_start"]).sort_values("date_start")[["date_start", "driver_number", "lap_number"]],
                              left_on="date", right_on="date_start", by="driver_number", direction="backward")
    else:
        radio["lap_number"] = None

    os.makedirs(CLIPS, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                             "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"}
    rows, ok = [], 0
    for i, r in radio.reset_index(drop=True).iterrows():
        fname = os.path.basename(r["recording_url"])
        # F1's server blocks many cloud hosts; FastF1's public mirror serves the same files
        urls = [r["recording_url"],
                r["recording_url"].replace("livetiming.formula1.com", "livetiming-mirror.fastf1.dev")]
        errors, saved = [], False
        for url in urls:
            try:
                resp = requests.get(url, headers=headers, timeout=60)
                if resp.status_code != 200 or len(resp.content) < 5000 or resp.content[:1] == b"<":
                    raise ValueError(f"HTTP {resp.status_code}, {len(resp.content)} bytes")
                with open(os.path.join(CLIPS, fname), "wb") as f:
                    f.write(resp.content)
                saved = True
                break
            except Exception as e:
                errors.append(f"{url.split('/')[2]}: {e}")
        if saved:
            ok += 1
        else:
            print(f"  clip {i} ({fname}) failed -> " + " | ".join(errors))
            fname = ""
        rows.append({"Driver": acr.get(r["driver_number"], r["driver_number"]),
                     "DriverNumber": r["driver_number"], "Time": r["date"], "Lap": r["lap_number"],
                     "File": fname, "AudioURL": r["recording_url"]})
    pd.DataFrame(rows).to_csv(os.path.join(CLIPS, "clips.csv"), index=False)
    print(f"Downloaded {ok}/{len(radio)} clips")
    if ok == 0:
        raise SystemExit("F1's server refused every clip from this machine.")
    shutil.make_archive("radio_clips", "zip", CLIPS)
    print(f"Created {os.path.abspath('radio_clips.zip')}")
    try:
        from IPython.display import FileLink, display
        display(FileLink("radio_clips.zip"))
    except Exception:
        pass


# ======================================================================= STEP 2: transcribe
def transcribe():
    if os.path.exists("radio_clips.zip"):
        with zipfile.ZipFile("radio_clips.zip") as z:
            z.extractall(CLIPS)
    if not os.path.isdir(CLIPS):
        raise SystemExit("Upload radio_clips.zip next to this notebook first.")

    # Build/refresh the clip list from the mp3 names, e.g. HAM_44_20261004_142600.mp3
    mp3s = sorted(f for f in os.listdir(CLIPS) if f.lower().endswith(".mp3"))
    if not mp3s:
        raise SystemExit(f"No mp3 files found in {CLIPS}/ - check radio_clips.zip.")
    rows = []
    for f in mp3s:
        # names: "Race__HAM_44_20261004_142600.mp3" (weekend download) or "HAM_44_..." (race-only download)
        session, base = (f.split("__", 1) if "__" in f else ("Race", f))
        parts = os.path.splitext(base)[0].split("_")
        t = pd.to_datetime(parts[2] + parts[3], format="%Y%m%d%H%M%S", errors="coerce") if len(parts) >= 4 else None
        rows.append({"Session": session, "Driver": parts[0],
                     "DriverNumber": parts[1] if len(parts) > 1 else None,
                     "Time": t, "File": f, "Base": base, "Tagged": "__" in f})
    listing = (pd.DataFrame(rows).sort_values("Tagged", ascending=False)
               .drop_duplicates("Base").drop(columns="Tagged"))   # same clip downloaded twice -> keep one
    old = os.path.join(CLIPS, "clips.csv")
    if os.path.exists(old):                       # keep lap numbers from step 1 if we have them
        prev = pd.read_csv(old)
        if "AudioURL" in prev.columns:
            prev["Base"] = prev["AudioURL"].astype(str).str.split("/").str[-1]
            listing = listing.merge(prev[["Base", "Lap", "AudioURL"]].drop_duplicates("Base"),
                                    on="Base", how="left")
    order = {"Practice1": 1, "Practice2": 2, "Practice3": 3, "SprintQualifying": 4,
             "Sprint": 5, "Qualifying": 6, "Race": 7}
    listing = (listing.assign(o=listing["Session"].map(order).fillna(9))
               .sort_values(["o", "Time"]).drop(columns=["o", "Base"]))
    print("Clips per session:", listing["Session"].value_counts().to_dict())
    listing.to_csv(old, index=False)

    if shutil.which("ffmpeg") is None:          # portable ffmpeg if the machine has none
        iio = pip_import("imageio-ffmpeg", "imageio_ffmpeg")
        bin_dir = os.path.join(os.path.expanduser("~"), ".ffmpeg_bin")
        os.makedirs(bin_dir, exist_ok=True)
        target = os.path.join(bin_dir, "ffmpeg.exe" if os.name == "nt" else "ffmpeg")
        if not os.path.exists(target):
            shutil.copy(iio.get_ffmpeg_exe(), target)
            os.chmod(target, 0o755)
        os.environ["PATH"] = bin_dir + os.pathsep + os.environ["PATH"]

    whisper = pip_import("openai-whisper", "whisper")
    model = whisper.load_model(MODEL)
    prompt = ("Formula 1 team radio. Box box, push, copy, mode push, overtake, DRS, deg, "
              "softs, mediums, hards, inters, safety car, VSC, gap, delta, undercut, "
              "lift and coast, battery, harvest, deploy, P1, Sepang, turn 1, turn 15.")

    clips = pd.read_csv(os.path.join(CLIPS, "clips.csv"))
    print(f"clips.csv: {len(clips)} rows | mp3 files in {CLIPS}/: "
          f"{sum(f.endswith('.mp3') for f in os.listdir(CLIPS))}")
    use_gpu = whisper.torch.cuda.is_available()
    print("GPU:", use_gpu)
    texts = []
    for i, r in clips.iterrows():
        text, note = "", ""
        path = os.path.join(CLIPS, str(r["File"])) if isinstance(r["File"], str) else ""
        if not path or not os.path.exists(path):
            note = f"(no audio file: {r['File']})"
        else:
            try:
                audio = whisper.load_audio(path)
                secs = len(audio) / 16000
                # radio audio is heavily compressed; disable Whisper's silence filter so speech isn't dropped
                opts = dict(language="en", fp16=use_gpu, no_speech_threshold=None,
                            logprob_threshold=None, condition_on_previous_text=False)
                text = model.transcribe(audio, initial_prompt=prompt, **opts)["text"].strip()
                if not text:                      # retry without the jargon prompt
                    text = model.transcribe(audio, **opts)["text"].strip()
                if not text:
                    note = f"(empty transcript, {secs:.1f}s audio, peak level {abs(audio).max():.3f})"
            except Exception as e:
                note = f"(failed: {str(e).strip().splitlines()[-1][:100]})"
        texts.append(text)
        print(f"  [{i + 1}/{len(clips)}] {r['Driver']}: {text[:70]} {note}")
    clips["Transcript"] = texts
    clips.drop(columns="File").to_csv(os.path.join(OUT, "team_radio.csv"), index=False)

    stop = set("""
    a an the and or but if so to of in on at for with from by as is are was were be been being it its
    this that these those i you he she we they me him her us them my your our their mine yours
    do does did done have has had will would can could should shall may might must just
    yeah yes no ok okay right um uh ah oh hmm mate guys guy got get go going gonna wanna
    there here what when where which who how all any some very really now then than too
    about up down out over again still well also not dont don't im i'm it's that's you're we're
    let lets let's one two three thing things bit like know see say said think
    """.split())
    wc = []
    sess_col = "Session" if "Session" in clips.columns else None
    keys = ["Driver", sess_col] if sess_col else ["Driver"]
    for key, g in clips.groupby(keys):
        drv, sess = (key if sess_col else (key[0] if isinstance(key, tuple) else key, None))
        words = re.findall(r"[a-zA-Z][a-zA-Z0-9']+", " ".join(g["Transcript"].fillna("")).lower())
        for w, n in Counter(w for w in words if w not in stop and len(w) > 2).items():
            row = {"Driver": drv, "Word": w, "Count": n}
            if sess_col:
                row["Session"] = sess
            wc.append(row)
    if not wc:
        raise SystemExit("\nNo words transcribed — paste the [n/N] lines above (with their notes) to Claude.")
    pd.DataFrame(wc).sort_values(["Driver", "Count"], ascending=[True, False]) \
        .to_csv(os.path.join(OUT, "radio_words.csv"), index=False)
    print(f"\nSaved {OUT}/team_radio.csv and {OUT}/radio_words.csv "
          f"({(clips['Transcript'] != '').sum()} clips transcribed)")


if STEP in ("download", "both"):
    download()
if STEP in ("transcribe", "both"):
    transcribe()

Clips per session: {'Race': 40, 'Qualifying': 18, 'Practice2': 9, 'Practice1': 7, 'Practice3': 4}
Installing openai-whisper (one-time)...


100%|████████████████████████████████████████| 461M/461M [00:03<00:00, 147MiB/s]


clips.csv: 78 rows | mp3 files in radio_clips/: 118
GPU: True
  [1/78] PIA: VSC, 
  [2/78] RUS: This is very cool. 
  [3/78] LAW: I have no brakes last mile. 
  [4/78] BOR: This is the craziest I've ever been in my life with oil temperatures,  
  [5/78] HAD: Wow, like zero front, it's coming up, zero. Okay, copy. 
  [6/78] HAD: Is that a free air place on this massive truck? This is probably the b 
  [7/78] RUS: Tell us how much we've got now, as we see it, four and eight, what abo 
  [8/78] LEC: VSC, 
  [9/78] GAS: I'm still a little concerned about this shower that's between 10 and 1 
  [10/78] VER: I feel sooo the car again, like this, second is a lot of you how I sit 
  [11/78] HAD: Ok, 2. 
  [12/78] ALB: We need to figure out, deploy, we need to turn 1. We need to figure ou 
  [13/78] BOR: What the f**k is this guy doing, man? He's seeing a cool up-down. I kn 
  [14/78] BOR: Set my floor early, water without the understood. 
  [15/78] BOR: Okay, what happened? Uh, there's another 